# SMA Crossover Strategy — Backtest & Walk-Forward Analysis

> **DISCLAIMER: NOT FINANCIAL ADVICE — EDUCATIONAL AND RESEARCH USE ONLY.**  
> This notebook demonstrates the ATS Research backtesting framework.  
> No live trading, no real money, no brokerage connections.

This notebook demonstrates:
1. Data loading and validation
2. Running a backtest with the SMA crossover strategy
3. Parameter sweep (grid) with walk-forward evaluation
4. Metrics analysis and overfitting checks

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from ats_research.data.loaders import generate_demo_data, load_ohlcv
from ats_research.data.validators import validate_ohlcv
from ats_research.data.features import sma, ema, atr, rsi
from ats_research.strategy.sma_crossover import SMACrossover
from ats_research.backtest.engine import BacktestEngine
from ats_research.backtest.metrics import compute_metrics, compute_drawdown_series
from ats_research.backtest.walkforward import WalkForwardSplitter, run_walkforward
from ats_research.utils.config import ATSConfig, load_config
from ats_research.utils.repro import set_global_seed
from ats_research.report.plots import (
    plot_equity_curve, plot_drawdown, plot_rolling_sharpe,
    plot_returns_distribution
)

print('ATS Research loaded successfully')

## 1. Data Loading & Validation

In [ ]:
# Generate synthetic demo data (or load from CSV)
set_global_seed(42)
data = generate_demo_data(n_bars=1000, seed=42)

print(f'Data shape: {data.shape}')
print(f'Date range: {data.index[0]} to {data.index[-1]}')
print(f'Columns: {list(data.columns)}')
data.head()

In [ ]:
# Validate the data
validated = validate_ohlcv(data)
print(f'Validated: {len(validated)} bars, no NaNs in OHLC')

# Compute some features
validated['sma_20'] = sma(validated['close'], 20)
validated['sma_50'] = sma(validated['close'], 50)
validated['atr_14'] = atr(validated, 14)
validated['rsi_14'] = rsi(validated['close'], 14)

# Plot price with SMAs
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

axes[0].plot(validated.index, validated['close'], label='Close', alpha=0.7)
axes[0].plot(validated.index, validated['sma_20'], label='SMA(20)', alpha=0.8)
axes[0].plot(validated.index, validated['sma_50'], label='SMA(50)', alpha=0.8)
axes[0].set_title('Price with Moving Averages')
axes[0].legend()

axes[1].plot(validated.index, validated['atr_14'], color='orange')
axes[1].set_title('ATR(14)')

axes[2].plot(validated.index, validated['rsi_14'], color='purple')
axes[2].axhline(70, color='red', linestyle='--', alpha=0.5)
axes[2].axhline(30, color='green', linestyle='--', alpha=0.5)
axes[2].set_title('RSI(14)')

plt.tight_layout()
plt.show()

## 2. Single Backtest Run

In [ ]:
# Load config
config = load_config('../config/example.yaml')
print(f'Strategy: {config.strategy.name}')
print(f'Params: {config.strategy.params}')
print(f'Risk: max_pos={config.risk.max_pos_pct}, trade_risk={config.risk.per_trade_risk_pct}')

In [ ]:
# Run backtest
set_global_seed(config.simulation.seed)
strategy = SMACrossover(params=config.strategy.params)
engine = BacktestEngine(config=config, strategy=strategy, data=data)
result = engine.run()

print(result.summary())

In [ ]:
# Plot equity curve and drawdown
fig1 = plot_equity_curve(result.equity_curve)
plt.show()

fig2 = plot_drawdown(result.equity_curve)
plt.show()

fig3 = plot_rolling_sharpe(result.equity_curve)
plt.show()

fig4 = plot_returns_distribution(result.equity_curve)
plt.show()

## 3. Parameter Sweep with Walk-Forward Analysis

We test multiple parameter combinations using walk-forward cross-validation
to assess robustness and detect overfitting.

In [ ]:
# Define parameter grid
param_grid = {
    'fast': [10, 20, 30],
    'slow': [40, 60, 80],
}

# Walk-forward splitter: 3 splits, 60% train, rolling window
splitter = WalkForwardSplitter(
    n_splits=3,
    train_pct=0.6,
    method='rolling',
    purge_days=5,
    embargo_days=2,
)

print('Running walk-forward analysis...')
wf_results = run_walkforward(
    config=config,
    strategy_cls=SMACrossover,
    param_grid=param_grid,
    data=data,
    splitter=splitter,
)
print(f'Completed: {len(wf_results)} results')
wf_results.head(10)

In [ ]:
# Aggregate results by parameter combination
summary = wf_results.groupby('params').agg({
    'train_sharpe': ['mean', 'std'],
    'test_sharpe': ['mean', 'std'],
    'train_cagr': 'mean',
    'test_cagr': 'mean',
    'train_max_dd': 'mean',
    'test_max_dd': 'mean',
}).round(4)

summary.columns = ['_'.join(col).strip('_') for col in summary.columns]
print('Walk-Forward Summary by Parameter Combination:')
summary

## 4. Overfitting Detection

Key checks:
- **Train vs Test gap**: Large gap in Sharpe/CAGR suggests overfitting
- **Test stability**: Low std in test Sharpe across splits = more robust
- **Turnover**: Very high turnover may indicate noise trading

In [ ]:
# Overfitting analysis
if not wf_results.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Train vs Test Sharpe
    avg_by_params = wf_results.groupby('params')[['train_sharpe', 'test_sharpe']].mean()
    avg_by_params.plot(kind='bar', ax=axes[0])
    axes[0].set_title('Train vs Test Sharpe Ratio (avg across splits)')
    axes[0].set_ylabel('Sharpe Ratio')
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].axhline(0, color='black', linestyle='-', alpha=0.3)
    
    # Train-Test gap
    avg_by_params['gap'] = avg_by_params['train_sharpe'] - avg_by_params['test_sharpe']
    avg_by_params['gap'].plot(kind='bar', ax=axes[1], color='coral')
    axes[1].set_title('Overfitting Gap (Train Sharpe - Test Sharpe)')
    axes[1].set_ylabel('Gap')
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].axhline(0, color='black', linestyle='-', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Summary statistics
    print('\nOverfitting Check:')
    mean_train = wf_results['train_sharpe'].mean()
    mean_test = wf_results['test_sharpe'].mean()
    print(f'  Avg Train Sharpe: {mean_train:.4f}')
    print(f'  Avg Test Sharpe:  {mean_test:.4f}')
    print(f'  Gap:              {mean_train - mean_test:.4f}')
    print(f'  Test Sharpe Std:  {wf_results["test_sharpe"].std():.4f}')
else:
    print('No walk-forward results to analyze')

In [ ]:
# Final metrics summary
print('='*60)
print('BACKTEST COMPLETE')
print('='*60)
print(f'\nMetrics from single run (fast={config.strategy.params["fast"]}, slow={config.strategy.params["slow"]}):')
for key, val in result.metrics.items():
    if isinstance(val, float):
        print(f'  {key:30s}: {val:>10.4f}')
    else:
        print(f'  {key:30s}: {val}')

print('\n--- NOT FINANCIAL ADVICE — EDUCATIONAL USE ONLY ---')